# TP CORRECTION — Module 2 : Feature Engineering Spatial
## Applications hydraulique et dégradation des terres — Correction annotée

> Chaque correction inclut :
> - Le code complet et fonctionnel
> - `# 💡` Justification du choix méthodologique
> - `# ⚠️` Piège courant et comment l'éviter
> - `# 📊` Lecture des résultats dans le contexte BF
> - `# ↔️ CM` Lien avec le concept correspondant dans le CM

## Environnement

In [ ]:
# !pip install geopandas libpysal esda rasterstats statsmodels folium
#             contextily requests scipy scikit-learn --quiet

---
## Correction A.1 — Features géométriques et hydrauliques

In [ ]:
# ── Correction A.1 ──

# 1. Features géométriques
gdf_utm = gdf.to_crs(epsg=32630)
gdf['superficie_km2'] = gdf_utm.geometry.area / 1e6
gdf['perimetre_km']   = gdf_utm.geometry.length / 1e3
gdf['compacite']      = (4 * np.pi * gdf_utm.geometry.area
                          / gdf_utm.geometry.length**2)
gdf['centroid_lat']   = gdf.geometry.centroid.y
# ↔️ CM §4.1 : même formule compacité — ici appliquée à l'accessibilité hydraulique

# 2. Couverture points d'eau
if 'pop' in gdf.columns and 'nb_puits' in gdf.columns:
    gdf['densite_pop']        = gdf['pop'] / gdf['superficie_km2'].clip(lower=0.1)
    gdf['puits_par_10000hab'] = (gdf['nb_puits'] / gdf['pop'].clip(lower=1)) * 10000
else:
    gdf['densite_pop']        = np.random.uniform(5, 900, len(gdf))
    gdf['puits_par_10000hab'] = np.random.uniform(0.5, 8, len(gdf))
# 💡 Standard DGRE-BF : 1 point d'eau fonctionnel pour 300 hab maximum
# → puits_par_10000hab > 3.3 = couverture correcte
# ⚠️ Ne pas diviser par pop sans clip(lower=1) → ZeroDivisionError

# 3. Population non desservie
if 'taux_acces_eau_pct' in gdf.columns:
    gdf['pop_non_desservie'] = gdf['pop'] * (1 - gdf['taux_acces_eau_pct'] / 100)
else:
    # Simulation : gradient latitudinal (moins bon accès au Nord)
    lat = gdf.geometry.centroid.y
    gdf['taux_acces_eau_pct'] = np.clip(88 - (lat - 9.5) * 4.5 +
                                         np.random.normal(0,5,len(gdf)), 20, 95)
    gdf['pop_non_desservie'] = gdf['pop'] * (1 - gdf['taux_acces_eau_pct'] / 100)

col = 'province' if 'province' in gdf.columns else gdf.columns[0]
print('=== Top 5 provinces par population non desservie ===')
print(gdf[[col,'taux_acces_eau_pct','pop_non_desservie','puits_par_10000hab']]
        .nlargest(5,'pop_non_desservie').to_string(index=False))

# 4. Carte taux d'accès à l'eau
fig, ax = plt.subplots(figsize=(10, 8))

# Palette tricolore : rouge < 40% | jaune 40-65% | vert > 65%
def couleur_acces(t):
    if t < 40:  return '#e74c3c'
    elif t < 65: return '#f1c40f'
    else:       return '#2ecc71'

gdf.plot(color=gdf['taux_acces_eau_pct'].apply(couleur_acces),
         edgecolor='white', linewidth=0.5, ax=ax)

legend_patches = [
    mpatches.Patch(color='#e74c3c', label='< 40% — Critique'),
    mpatches.Patch(color='#f1c40f', label='40–65% — Insuffisant'),
    mpatches.Patch(color='#2ecc71', label='> 65% — Acceptable'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=10)
ax.set_title('Taux d\'accès à l\'eau potable — Burkina Faso\n(données DGRE 2023)',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('TP_A1_acces_eau.png', dpi=150, bbox_inches='tight')
plt.show()

# 📊 Résultat attendu : les provinces sahéliennes (Oudalan, Séno) ont les taux
# les plus faibles — elles seront aussi les plus vulnérables dans l'IVH final.

---
## Correction A.2 — Voisinage hydraulique

In [ ]:
# ── Correction A.2 ──

# 1. Matrice W + gestion îles
acces_vals = gdf['taux_acces_eau_pct'].fillna(gdf['taux_acces_eau_pct'].median()).values

W = Queen.from_dataframe(gdf, use_index=True)
W.transform = 'r'
if W.islands:
    W = KNN.from_dataframe(gdf, k=4); W.transform = 'r'
print(f'W construite : {W.n} provinces | moy voisins : {W.mean_neighbors:.1f}')

# 2. Moran I de l'accès à l'eau
from esda.moran import Moran, Moran_Local
moran_eau = Moran(acces_vals, W)
print(f'\n=== Moran I — Taux accès eau BF ===')
print(f'I = {moran_eau.I:.4f} | p = {moran_eau.p_sim:.4f}')
# ↔️ CM §5.4 : même calcul — mais variable hydraulique au lieu d'IPC
# 📊 Attendu : I > 0 (bon accès au Sud, mauvais au Nord → clustering positif)
# Comparer avec Moran I IPC du CM : les deux devraient être positifs et significatifs

# 3. Lag spatial de l'accès
gdf['acces_lag'] = lag_spatial(W, acces_vals)

# 4. LISA
lisa_eau = Moran_Local(acces_vals, W, seed=42)
gdf['acces_lisa_q']   = lisa_eau.q
gdf['acces_lisa_p']   = lisa_eau.p_sim
gdf['acces_lisa_cat'] = gdf.apply(
    lambda r: {1:'HH',2:'LH',3:'LL',4:'HL'}.get(int(r['acces_lisa_q']),'NS')
              if r['acces_lisa_p'] < 0.05 else 'NS', axis=1)
print('\nLISA accès eau :')
print(gdf['acces_lisa_cat'].value_counts())
print('LL = clusters de mauvais accès = zones prioritaires DGRE')

# 5. Carte LISA
palette_lisa = {'HH':'#2ecc71','LL':'#e74c3c','HL':'#e67e22','LH':'#3498db','NS':'#bdc3c7'}
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(color=gdf['acces_lisa_cat'].map(palette_lisa),
         edgecolor='white', linewidth=0.5, ax=ax)
leg = [mpatches.Patch(color=v,label={'HH':'HH — Cluster bon accès','LL':'LL — Cluster déficit',
       'HL':'HL — Bonne province entourée de mauvaises','LH':'LH — Mauvaise entourée de bonnes',
       'NS':'Non significatif'}[k]) for k,v in palette_lisa.items()]
ax.legend(handles=leg, loc='lower left', fontsize=8)
ax.set_title(f'LISA — Accès eau BF · Moran I = {moran_eau.I:.3f} (p={moran_eau.p_sim:.3f})',
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('TP_A2_lisa_eau.png', dpi=150, bbox_inches='tight')
plt.show()

# Réponse analytique :
# Les zones LL (mauvais accès + voisines mauvais accès) correspondent généralement
# aux zones HH (IPC élevé) du CM. Cela reflète le lien eau-sécurité alimentaire :
# sans eau potable, les activités agricoles et d'élevage sont compromises.
# ⚠️ Attention : LL ici = mauvais accès (à identifier pour intervention),
# alors que LL en IPC = bonne situation. Le signe LISA dépend de la direction
# de la variable — vérifier toujours.

---
## Correction A.3 — Indice IVH et carte Folium

In [ ]:
# ── Correction A.3 ──

# 1. Indice composite IVH
scaler = MinMaxScaler()
composantes = ['taux_acces_eau_pct','puits_par_10000hab','acces_lag','compacite']
composantes_ok = [c for c in composantes if c in gdf.columns]
X_comp = gdf[composantes_ok].fillna(gdf[composantes_ok].median())
X_norm = scaler.fit_transform(X_comp)
df_n = pd.DataFrame(X_norm, columns=[c+'_n' for c in composantes_ok])

# IVH : poids définis par la DGRE — fort poids sur l'accès direct
gdf['IVH'] = 0.0
weights = {'taux_acces_eau_pct_n':0.40,'puits_par_10000hab_n':0.30,
           'acces_lag_n':0.20,'compacite_n':0.10}
for col_n, w in weights.items():
    if col_n in df_n.columns:
        if col_n in ['taux_acces_eau_pct_n','puits_par_10000hab_n','acces_lag_n']:
            gdf['IVH'] += w * (1 - df_n[col_n].values)  # inverser : plus c'est bas, plus vulnérable
        else:  # compacite : inverser aussi
            gdf['IVH'] += w * (1 - df_n[col_n].values)
# 💡 On inverse taux_acces et compacite car IVH = vulnérabilité (élevé = mauvais)

# 2. Top-5
col = 'province' if 'province' in gdf.columns else gdf.columns[0]
top5 = gdf[[col,'region' if 'region' in gdf.columns else gdf.columns[1],
             'IVH','taux_acces_eau_pct','acces_lisa_cat']].nlargest(5,'IVH')
print('=== Top-5 provinces vulnérabilité hydraulique ===')
print(top5.to_string(index=False))

# 3. VIF des composantes
X_vif = df_n[[c for c in df_n.columns if c in
              [k for k in weights.keys() if k in df_n.columns]]]
if X_vif.shape[1] >= 2:
    vif_comp = pd.DataFrame({
        'Feature': X_vif.columns,
        'VIF': [variance_inflation_factor(X_vif.values, i)
                for i in range(X_vif.shape[1])]
    }).sort_values('VIF',ascending=False)
    print('\nVIF des composantes IVH :')
    print(vif_comp.to_string(index=False))
    # 📊 Attendu : taux_acces et acces_lag corrélés (VIF > 2) mais < 10
    # → redondance partielle acceptable pour un indice composite

# 4. Carte Folium
m = folium.Map(location=[12.37, -1.54], zoom_start=6, tiles='CartoDB positron')
geojson_str = gdf[[col,'IVH','taux_acces_eau_pct','acces_lisa_cat','geometry']].to_json()

folium.Choropleth(
    geo_data=geojson_str,
    data=gdf,
    columns=[col,'IVH'],
    key_on=f'feature.properties.{col}',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.8,
    legend_name='Indice de Vulnérabilité Hydraulique (IVH)',
).add_to(m)

folium.GeoJson(
    geojson_str,
    style_function=lambda x: {'fillOpacity':0,'color':'transparent'},
    tooltip=folium.GeoJsonTooltip(
        fields=[col,'IVH','taux_acces_eau_pct','acces_lisa_cat'],
        aliases=['Province :','IVH :','Accès eau % :','LISA :'],
        localize=True
    )
).add_to(m)

m.save('TP_A3_IVH_interactive.html')
print('Carte IVH interactive sauvegardée.')

---
## Correction B.1 — Analyse changement végétation

In [ ]:
# ── Correction B.1 ──

# 1. Delta NDVI
if 'ndvi_2023' in gdf.columns and 'ndvi_2020' in gdf.columns:
    gdf['delta_ndvi'] = gdf['ndvi_2023'] - gdf['ndvi_2020']
    # Taux de changement normalisé par la valeur de référence
    gdf['taux_changement_pct'] = ((gdf['delta_ndvi'] / gdf['ndvi_2020'].clip(lower=0.01)) * 100)
else:
    # Simulation : dégradation progressive Nord→Centre
    lat = gdf.geometry.centroid.y
    np.random.seed(99)
    gdf['delta_ndvi'] = -0.04 + (lat - 9.5) * (-0.008) + np.random.normal(0, 0.01, len(gdf))
    gdf['taux_changement_pct'] = gdf['delta_ndvi'] / 0.20 * 100
# ↔️ CM §7.3 : la tendance FAPAR mesure la même chose sur 12 mois
# Ici on a 2 observations (2020 et 2023) — moins précis mais suffisant pour détecter des tendances

# 2. Classification
def classe_change(t):
    if t < -15:  return 'Dégradation sévère'
    elif t < -5: return 'Dégradation modérée'
    elif t < 5:  return 'Stable'
    else:        return 'Amélioration'

gdf['classe_degradation'] = gdf['taux_changement_pct'].apply(classe_change)
print('Répartition des classes :')
print(gdf['classe_degradation'].value_counts())

# 3. Graphique à barres horizontales
gdf_sorted = gdf.sort_values('delta_ndvi').copy()
col = 'province' if 'province' in gdf.columns else gdf.columns[0]

palette_deg = {
    'Dégradation sévère':'#e74c3c',
    'Dégradation modérée':'#e67e22',
    'Stable':'#95a5a6',
    'Amélioration':'#2ecc71'
}
colors = gdf_sorted['classe_degradation'].map(palette_deg)

fig, ax = plt.subplots(figsize=(10, max(8, len(gdf_sorted)*0.35)))
bars = ax.barh(gdf_sorted[col].str[:12], gdf_sorted['delta_ndvi'],
               color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1.5, linestyle='--', alpha=0.7)
ax.set_xlabel('Delta NDVI (2023 − 2020)', fontsize=11)
ax.set_title('Évolution de la végétation 2020–2023\nBurkina Faso (données Sentinel-2)',
             fontsize=11, fontweight='bold')
leg = [mpatches.Patch(color=v,label=k) for k,v in palette_deg.items()]
ax.legend(handles=leg, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('TP_B1_delta_ndvi.png', dpi=150, bbox_inches='tight')
plt.show()
# ⚠️ Utiliser le taux_changement_pct pour comparer les provinces équitablement.
# La province Sahel avec ndvi_2020=0.06 et delta=-0.01 perd 17% de sa maigre végétation.
# La province Cascades avec ndvi_2020=0.46 et delta=-0.01 perd seulement 2%.
# En delta absolu, elles semblent identiques. Le taux révèle la différence.

---
## Correction B.2 — Structure spatiale dégradation

In [ ]:
# ── Correction B.2 ──

delta_vals = gdf['delta_ndvi'].fillna(0).values

# 1. Moran I
moran_deg = Moran(delta_vals, W)
print(f'Moran I delta_ndvi = {moran_deg.I:.4f} | p = {moran_deg.p_sim:.4f}')
if moran_deg.I < 0 and moran_deg.p_sim < 0.05:
    print('→ Pattern en damier : dégradation et amélioration alternent (inhabituel)')
elif moran_deg.I > 0 and moran_deg.p_sim < 0.05:
    print('→ Clustering : zones dégradées contiguës = front de dégradation détecté')

# 2. Lag spatial
gdf['delta_ndvi_lag'] = lag_spatial(W, delta_vals)

# Interprétation différentielle :
# delta_ndvi=-0.03 + lag=-0.02 → province dégradée entourée de provinces dégradées
#   → front régional, situation systémique, intervention à grande échelle nécessaire
# delta_ndvi=-0.03 + lag=+0.01 → province dégradée entourée de provinces stables
#   → choc local (surpâturage, déforestation locale), intervention ciblée possible

# 3. LISA dégradation
lisa_deg = Moran_Local(delta_vals, W, seed=42)
gdf['deg_lisa_q']   = lisa_deg.q
gdf['deg_lisa_p']   = lisa_deg.p_sim
gdf['deg_lisa_cat'] = gdf.apply(
    lambda r: {1:'HH',2:'LH',3:'LL',4:'HL'}.get(int(r['deg_lisa_q']),'NS')
              if r['deg_lisa_p'] < 0.05 else 'NS', axis=1)
# ⚠️ Attention au signe : delta_ndvi positif = amélioration
# → HH = cluster de provinces QUI S'AMÉLIORENT (bon)
# → LL = cluster de provinces QUI SE DÉGRADENT (mauvais = front de désertification)
print('\nLISA dégradation :')
print(gdf['deg_lisa_cat'].value_counts())
print('LL = front de désertification | HH = zone de reboisement/verdissement')

# 4. Cartes côte à côte
palette_lisa_deg = {'HH':'#2ecc71','LL':'#8e44ad','HL':'#e67e22','LH':'#3498db','NS':'#bdc3c7'}
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Carte LISA
gdf.plot(color=gdf['deg_lisa_cat'].map(palette_lisa_deg),
         edgecolor='white', linewidth=0.5, ax=axes[0])
axes[0].set_title(f'LISA — Changement végétation\nMoran I = {moran_deg.I:.3f}',
                  fontsize=11, fontweight='bold')
axes[0].axis('off')
leg_d = [mpatches.Patch(color=v,label={'HH':'HH — Cluster verdissement','LL':'LL — Front dégradation',
         'HL':'HL — Dégradée isolée','LH':'LH — Stable isolée','NS':'NS'}[k])
         for k,v in palette_lisa_deg.items()]
axes[0].legend(handles=leg_d, loc='lower left', fontsize=8)

# Carte delta NDVI
gdf.plot(column='delta_ndvi', cmap='RdYlGn', legend=True,
         legend_kwds={'label':'Delta NDVI','orientation':'vertical'},
         edgecolor='white', linewidth=0.5, ax=axes[1])
axes[1].set_title('Delta NDVI (2023-2020)\nRouge=dégradation · Vert=amélioration',
                  fontsize=11, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('TP_B2_spatial_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Correction B.3 — Rapport d'alerte précoce

In [ ]:
# ── Correction B.3 ──

# 1. Feature matrix dégradation
FEATS_DEG = ['delta_ndvi','taux_changement_pct','delta_ndvi_lag',
             'pluvio_mm' if 'pluvio_mm' in gdf.columns else 'centroid_lat',
             'densite_pop','superficie_km2']
feats_ok = [f for f in FEATS_DEG if f in gdf.columns]
X_deg = gdf[feats_ok].dropna()
print(f'Feature matrix dégradation : {X_deg.shape[0]} provinces x {len(feats_ok)} features')

# 2. VIF
X_deg_norm = MinMaxScaler().fit_transform(X_deg)
vif_deg = pd.DataFrame({
    'Feature': feats_ok,
    'VIF': [variance_inflation_factor(X_deg_norm, i) for i in range(X_deg_norm.shape[1])]
}).sort_values('VIF', ascending=False)
print('\nVIF — Features dégradation :')
print(vif_deg.to_string(index=False))
# 📊 delta_ndvi et delta_ndvi_lag seront corrélés (VIF 2-5) — acceptable
# si VIF > 10 : retirer delta_ndvi_lag et conserver delta_ndvi (plus direct)

# 3. Top-5 provinces en alerte
# Critère combiné : forte dégradation locale ET voisinage dégradé
col = 'province' if 'province' in gdf.columns else gdf.columns[0]
gdf['score_alerte'] = -(gdf['taux_changement_pct'].fillna(0)  # plus négatif = pire
                         + gdf['delta_ndvi_lag'].fillna(0) * 100)  # voisinage dégradé amplifie

top5_b = gdf[[col,'score_alerte','taux_changement_pct','delta_ndvi_lag','deg_lisa_cat']]\
           .nlargest(5,'score_alerte')
print('\n=== Top-5 provinces en alerte dégradation ===')
print(top5_b.to_string(index=False))

# 4. Analyse qualitative et question de synthèse
# Les provinces prioritaires seront celles du Sahel (Oudalan, Séno, Soum) :
# - Dégradation sévère : ndvi_2020 déjà bas, toute perte supplémentaire est critique
# - Voisinage dégradé : front de désertification régional, pas un choc isolé
# - Interventions recommandées : zaï (agriculture sur sol dégradé), cordons pierreux,
#   mise en défens pastoral, reboisement Prosopis/Acacia

# Question synthèse — Double crise eau + terres :
if 'IVH' in gdf.columns:
    gdf['score_double_crise'] = gdf['IVH'].fillna(0) + gdf['score_alerte'].fillna(0)/100
    top3_double = gdf[[col,'IVH','score_alerte','score_double_crise']].nlargest(3,'score_double_crise')
    print('\n=== Provinces en double crise eau + terres ===')
    print(top3_double.to_string(index=False))
    print('\nCes provinces cumulent vulnérabilité hydraulique ET dégradation végétale.')
    print('Elles représentent les zones à plus fort risque de crise alimentaire future.')
    print('Recommandation : intervention intégrée WASH + restauration des terres.')

---
## Synthèse des corrections — Module 2 TP v4

### Pourquoi ces exercices sont différents du CM

| | CM (cours) | TP v4 (ces exercices) |
|-|-----------|----------------------|
| **Variable cible** | ipc_phase (insécurité alimentaire 1–5) | taux_acces_eau + delta_ndvi |
| **Question** | Quelles features prédisent l'IPC ? | Où investir en hydraulique ? Où est la dégradation ? |
| **Résultat** | Feature matrix → input ML Module 3 | Indices IVH + alertes précoces → aide à la décision |
| **Approche** | Exploratoire (construire les features) | Décisionnelle (répondre à des questions) |

### Ce que ce TP évalue

| Compétence | Comment évaluée |
|-----------|----------------|
| Géométrie + reprojection | A.1 : compacité sur vraies géométries GADM |
| Lag spatial + LISA | A.2 et B.2 : sur deux variables différentes du CM |
| Interprétation du signe LISA | A.2 : LL hydraulique ≠ LL IPC (sens inversé) |
| VIF sur indice composite | A.3 : composantes de l'IVH |
| Taux vs valeur absolue | B.1 : normalisation par la valeur de référence |
| Synthèse multi-critères | B.3 : double crise eau + terres |